In [1]:
# Parameters
execution_date = "2025-02-23"


In [2]:
import duckdb
import requests
import pandas as pd

In [3]:
conn = duckdb.connect("/Users/andrebezerra/Desktop/Dev/data_with_duckdb/v2_database.duckdb")
df_gen = conn.execute("select * from generations").fetch_df()
conn.close()

# Filter the first row where 'status' is blank (NaN or empty string)
run_gen = df_gen[df_gen['STATUS'].isna() | (df_gen['STATUS'] == '')].iloc[0]['NAME']

# Show value 'name'
print(f'Running for Gen: {run_gen}')

Running for Gen: generation-i


In [4]:
# URL da API para a geração específica
url = f"https://pokeapi.co/api/v2/generation/{run_gen}/"

# Requisição para obter os dados da geração
response = requests.get(url)

# Verificar se a requisição foi bem-sucedida
if response.status_code == 200:
    data = response.json()
    
    # Extrair a lista de Pokémon da geração
    pokemon_species = data.get("pokemon_species", [])
    
    # Extrair o nome e o ID do Pokémon a partir da URL
    pokemon_data = [
        {
            "id": int(pokemon["url"].rstrip("/").split("/")[-1])  # Extrair o ID da URL
        }
        for pokemon in pokemon_species
    ]
    
    # Criar o DataFrame
    df_ids = pd.DataFrame(pokemon_data)
    df_ids = df_ids.sort_values(by='id', ascending=True)

    # Obter o número de linhas e colunas do DataFrame
    n_rows, n_cols = df_ids.shape

    # Exibir o resultado
    print(f"Dataframe has {n_rows} rows and {n_cols} columns.")
    
else:
    print(f"Erro ao acessar a API: {response.status_code}")

Dataframe has 151 rows and 1 columns.


In [5]:
# URL base da PokeAPI
api_url = "https://pokeapi.co/api/v2/pokemon/"

# Função para buscar informações de tipos usando um DataFrame de IDs
def fetch_pokemon_types(df_ids):
    detailed_data = []

    # Iterar sobre a coluna 'id' do DataFrame
    for pokemon_id in df_ids['id']:
        response = requests.get(f"{api_url}{pokemon_id}")
        if response.status_code == 200:
            data = response.json()
            types = data.get("types", [])
            
            # Inicializando os tipos
            type_1 = None
            type_2 = None
            
            # Iterando nos tipos para extrair conforme o slot
            for type_info in types:
                if type_info["slot"] == 1:
                    type_1 = type_info["type"]["name"]
                elif type_info["slot"] == 2:
                    type_2 = type_info["type"]["name"]
            
            # Adiciona ao dataset detalhado
            detailed_data.append({
                "id": pokemon_id,
                "type_1": type_1,
                "type_2": type_2
            })
        else:
            print(f"Erro ao acessar detalhes do Pokémon ID {pokemon_id}: {response.status_code}")

    # Retornar os dados em um DataFrame
    return pd.DataFrame(detailed_data)

# Buscar informações de tipos
df_types = fetch_pokemon_types(df_ids)

In [6]:
# Create DuckDB database
conn = duckdb.connect("/Users/andrebezerra/Desktop/Dev/data_with_duckdb/v2_database.duckdb")
conn.execute("""
    CREATE TABLE IF NOT EXISTS b_types ( 
      id INT,
      type_1 TEXT,
      type_2 TEXT
    )
""")
conn.execute("""INSERT INTO b_types SELECT * FROM df_types""")
conn.close()